In [1]:
import torch

In [ ]:
# 归一化的函数
def normalize_to_minus1_1(x: torch.Tensor, min_val, max_val) -> torch.Tensor:
    """
    将输入按区间 [min_val, max_val] 线性映射到 [-1, 1] 并裁剪：
    - x <= min_val -> -1
    - x >= max_val -> 1
    - 其余线性映射到 (-1, 1)
    当 max_val == min_val 时（退化区间）：
    - x < min_val -> -1, x > max_val -> 1, 等于 -> 0

    min_val / max_val 可为标量或与 x 可广播的张量。
    """
    min_t = torch.as_tensor(min_val, dtype=x.dtype, device=x.device)
    max_t = torch.as_tensor(max_val, dtype=x.dtype, device=x.device)
    denom = max_t - min_t
    # 避免除零：仅对非退化位置执行标准线性映射
    denom_safe = torch.where(denom == 0, torch.ones_like(denom), denom)
    y = (x - min_t) / denom_safe
    y = y * 2 - 1
    # 裁剪到 [-1, 1]
    y = torch.clamp(y, -1.0, 1.0)
    # 退化处理：max==min
    deg_mask = (denom == 0)
    if torch.any(deg_mask):
        y_degen = torch.where(
            x > max_t, torch.ones_like(x),
            torch.where(x < min_t, -torch.ones_like(x), torch.zeros_like(x))
        )
        y = torch.where(deg_mask, y_degen, y)
    return y

In [4]:
x = [1,2,3,4,5,6,7,8,9,10]
x = torch.tensor(x)
print(normalize_to_minus1_1(x, 1, 10))

tensor([-1.0000, -0.7778, -0.5556, -0.3333, -0.1111,  0.1111,  0.3333,  0.5556,
         0.7778,  1.0000])
